# Level 3 — Skin wound CellCharter spatial niches

Spatial niche detection for the wound-skin dataset, mirroring the tumor `04_cellcharter` notebook. Pipeline: scVI batch correction (`batch=sample_id`) → Delaunay spatial neighbors → CellCharter neighborhood aggregation → GMM niche sweep (k=8/10/12, best by BIC).

Input: `../../data/talbot_xenium_skin_annotated.h5ad` (already CyteType-annotated) plus raw counts from `../../data/talbot_xenium.h5ad`.

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import squidpy as sq
import cellcharter as cc
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib import colors as mcolors
from sklearn.mixture import GaussianMixture
from lightning.pytorch import seed_everything
import warnings; warnings.filterwarnings('ignore')

seed_everything(12345)
scvi.settings.seed = 12345

In [ ]:
KS = [8, 10, 12]
ANNOT = 'cytetype_annotation_leiden_4'

## Load annotated skin + raw counts

In [ ]:
adata = sc.read_h5ad('../../data/talbot_xenium_skin_annotated.h5ad')
raw = sc.read_h5ad('../../data/talbot_xenium.h5ad')
raw = raw[raw.obs.tissue != 'tumor']
adata.layers['counts'] = raw[adata.obs_names].X.copy()
adata

## scVI batch correction (batch = sample_id)

In [ ]:
scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key='sample_id')
model = scvi.model.SCVI(adata)

In [ ]:
model.train(early_stopping=True, enable_progress_bar=True)

In [ ]:
adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)

## Spatial neighbors + CellCharter aggregation

In [ ]:
sq.gr.spatial_neighbors(adata, library_key='sample_id', coord_type='generic',
                        delaunay=True, percentile=99)

In [ ]:
cc.gr.aggregate_neighbors(adata, n_layers=3, use_rep='X_scVI',
                          out_key='X_cellcharter', sample_key='sample_id')

## GMM niche sweep (k = 8, 10, 12)

In [ ]:
X = adata.obsm['X_cellcharter']
rows = []
for k in KS:
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=0, n_init=3)
    labels = gmm.fit_predict(X)
    adata.obs[f'CellCharter_{k}'] = pd.Categorical(labels.astype(str))
    rows.append({'k': k, 'bic': gmm.bic(X), 'aic': gmm.aic(X)})
    print(f'k={k}: bic={rows[-1]["bic"]:.0f}  aic={rows[-1]["aic"]:.0f}')
sweep = pd.DataFrame(rows)
best_k = int(sweep.loc[sweep.bic.idxmin(), 'k'])
print('best k by BIC =', best_k)
os.makedirs('../../results/cell-driven-analysis', exist_ok=True)
sweep.to_csv('../../results/cell-driven-analysis/skin_wound_cellcharter_gmm_sweep.csv', index=False)

## Save niche-annotated object

In [ ]:
adata.write('../../data/talbot_xenium_skin_annotated_cellcharter.h5ad')

## Per-condition niche spatial maps

In [ ]:
def plot_by_condition(ad, color, out_stem, spot_size=2.0, cols=3, title=None):
    coords = np.asarray(ad.obsm['spatial'])[:, :2]
    cats = ad.obs[color].astype('category')
    names = list(cats.cat.categories); codes = cats.cat.codes.to_numpy()
    base = list(plt.get_cmap('tab20').colors)
    col_list = (base * int(np.ceil(len(names)/len(base))))[:len(names)]
    rgba = np.array([mcolors.to_rgba(c) for c in col_list])
    carr = np.zeros((codes.size, 4)); carr[codes >= 0] = rgba[codes[codes >= 0]]
    conds = [c for c in ['litt_24h','cre_24h','wt_24h','litt_72h','cre_72h','wt_72h']
             if c in set(ad.obs['condition'].astype(str))]
    g = ad.obs['condition'].astype(str).to_numpy()
    r_ = int(np.ceil(len(conds)/cols))
    fig = plt.figure(figsize=(4.5*cols+3, 4.5*r_), dpi=150)
    gs = GridSpec(r_, cols+1, figure=fig, width_ratios=[1]*cols+[1.2], wspace=0.03, hspace=0.1)
    for i, cond in enumerate(conds):
        rr, cc_ = divmod(i, cols); ax = fig.add_subplot(gs[rr, cc_])
        idx = np.flatnonzero(g == cond); xy = coords[idx]
        ax.scatter(xy[:,0], xy[:,1], c=carr[idx], s=spot_size, marker='o', linewidths=0, rasterized=True)
        ax.set_title(cond, fontsize=11); ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_axis_off()
    for j in range(len(conds), r_*cols):
        rr, cc_ = divmod(j, cols); fig.add_subplot(gs[rr, cc_]).axis('off')
    axl = fig.add_subplot(gs[:, -1]); axl.axis('off')
    h = [Line2D([0],[0], marker='o', color='w', markerfacecolor=col_list[k], markersize=8, label=str(c))
         for k, c in enumerate(names)]
    axl.legend(handles=h, title=color, frameon=False, loc='center left', fontsize=9, title_fontsize=10)
    if title: fig.suptitle(title, fontsize=14, y=0.995)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.95, bottom=0.02)
    os.makedirs('../../results/spatial-tissue-maps', exist_ok=True)
    for ext in ('png','pdf'):
        fig.savefig(f'{out_stem}.{ext}', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
for k in KS:
    plot_by_condition(adata, f'CellCharter_{k}',
        f'../../results/spatial-tissue-maps/skin_wound_niches_k{k}_by_condition',
        title=f'Wound skin — CellCharter niches (k={k}) by condition')

## Niche CSV exports (per-cell assignments + niche × cell-type composition)

In [ ]:
best_col = f'CellCharter_{best_k}'
sp = np.asarray(adata.obsm['spatial'])[:, :2]
per_cell = pd.DataFrame({
    'cell_id': adata.obs_names, 'x': sp[:,0], 'y': sp[:,1],
    'sample_id': adata.obs['sample_id'].astype(str).values,
    'condition': adata.obs['condition'].astype(str).values,
    'genotype': adata.obs['genotype'].astype(str).values,
    'cell_type': adata.obs[ANNOT].astype(str).values,
})
for k in KS:
    per_cell[f'niche_k{k}'] = adata.obs[f'CellCharter_{k}'].astype(str).values
per_cell.to_csv('../../results/cell-driven-analysis/skin_wound_cell_niches.csv', index=False)
comp = pd.crosstab(adata.obs[best_col], adata.obs[ANNOT], normalize='index')
comp.to_csv(f'../../results/cell-driven-analysis/skin_wound_niche_composition_k{best_k}.csv')
print('done; best_k =', best_k)